# Run-StreamingLoad (Fabric Notebook)

Continuous streaming generator for `WorkspaceLogs` - simulates live Fabric item events for streaming-load / soak tests.

Mirrors the logic of `Run-StreamingLoad.ps1` so it can be **scheduled** in Fabric.

**Endpoint:** `POST {QueryUri}/v1/rest/ingest/{db}/WorkspaceLogs?streamFormat=JSON&mappingName=WorkspaceLogs_mapping`

Distributions:
* type: 80% `Microsoft.Fabric.ItemCreateSucceeded` / 20% `Microsoft.Fabric.ItemDeleteSucceeded`
* `data_itemKind`: chosen from a list keyed off the workspace's `DataverseWorkspace.OAPActivity` value
* Plus 0-1 "flagged" Notebook events injected per HTTP batch (any workspace).

**Prereqs:** streaming ingestion enabled on database AND `WorkspaceLogs` table:
```kusto
.alter database <db> policy streamingingestion enable
.alter table  WorkspaceLogs policy streamingingestion enable
```

## Parameters

In [ ]:
# ---- Parameters (override at schedule time) ---------------------------------
QueryUri          = 'https://trd-6uegjpfbf030eemxtw.z1.kusto.fabric.microsoft.com'
DatabaseName      = 'MonitoringEventhouse'

LoadProfile       = 'steady'      # 'steady' | 'burst' | 'ramp'
RatePerSecond     = 100
BurstRate         = 1000
BurstSeconds      = 30
IdleSeconds       = 60
DurationMinutes   = 60            # run for 1 hour
BatchSize         = 200
WorkspacePoolSize = 20000
FlaggedProb       = 0.5           # P(extra flagged Notebook event per batch)

## Acquire Kusto token

Uses the notebook's run identity (user, service principal, or workspace identity when scheduled).

In [ ]:
import notebookutils

# Fabric provides a 'kusto' audience for Eventhouse REST.
kql_token = notebookutils.credentials.getToken('kusto')
print('Acquired Kusto token (len=%d).' % len(kql_token))

## Load workspace pool joined with `DataverseWorkspace.OAPActivity`

In [ ]:
import json, requests, random, time, uuid
from datetime import datetime, timezone, timedelta

query_uri  = f"{QueryUri}/v1/rest/query"
ingest_uri = (
    f"{QueryUri}/v1/rest/ingest/{DatabaseName}/WorkspaceLogs"
    "?streamFormat=JSON&mappingName=WorkspaceLogs_mapping"
)

json_headers = {
    'Authorization': f'Bearer {kql_token}',
    'Content-Type':  'application/json',
}
ingest_headers = dict(json_headers)

pool_kql = f"""
_TestWorkspacePool
| take {WorkspacePoolSize}
| join kind=leftouter (
    DataverseWorkspace
    | summarize arg_max(ingestion_time(), OAPActivity) by WorkspaceId
  ) on WorkspaceId
| project WorkspaceId, WorkspaceName, OAPActivity
"""

resp = requests.post(
    query_uri,
    headers=json_headers,
    json={'csl': pool_kql, 'db': DatabaseName},
    timeout=600,
)
resp.raise_for_status()
tables = resp.json().get('Tables', [])
primary = next((t for t in tables if t.get('TableName') in ('Table_0', 'PrimaryResult')), None)
if not primary or not primary.get('Rows'):
    raise RuntimeError('_TestWorkspacePool is empty. Run the backfill -Phases Pool first.')

pool = [
    {'WorkspaceId': r[0], 'WorkspaceName': r[1], 'OAPActivity': r[2]}
    for r in primary['Rows']
]
pool_count = len(pool)
enabled  = sum(1 for w in pool if w['OAPActivity'] == 'EnableWorkspaceOutboundAccessProtection')
disabled = sum(1 for w in pool if w['OAPActivity'] == 'DisableWorkspaceOutboundAccessProtection')
print(f'Loaded {pool_count} workspaces (OAP enabled: {enabled}, disabled: {disabled}, other/null: {pool_count - enabled - disabled}).')

## Event generator

In [ ]:
ITEM_KINDS_OAP_ENABLED = [
    'Lakehouse', 'Notebook', 'SparkJobDefinition', 'Environment', 'Warehouse',
    'DataFlow', 'DataPipeline', 'CopyJob', 'MirroredDatabase', 'SQLEndpoint',
    'VariableLibrary', 'SqlAnalyticsEndpoint', 'Experiment', 'MlModel',
]
ITEM_KINDS_OAP_DISABLED = [
    'VariableLibrary', 'SemanticModel', 'Report', 'App', 'Dashboard', 'Scorecard',
    'KQLDashboard', 'Eventstream', 'cosmosdb', 'azuredb', 'SqlAnalyticsEndpoint',
    'MountedDataFactory', 'DataFlowGen1',
]
ITEM_KINDS_ANY = list(dict.fromkeys(ITEM_KINDS_OAP_ENABLED + ITEM_KINDS_OAP_DISABLED))

rng = random.Random()

def new_event(force_kind=None):
    ws = pool[rng.randrange(pool_count)]
    is_create = rng.random() < 0.8
    ev_type = 'Microsoft.Fabric.ItemCreateSucceeded' if is_create else 'Microsoft.Fabric.ItemDeleteSucceeded'
    item_id = str(uuid.uuid4())

    if force_kind:
        kind = force_kind
    else:
        oap = ws['OAPActivity']
        if oap == 'EnableWorkspaceOutboundAccessProtection':
            kind_list = ITEM_KINDS_OAP_ENABLED
        elif oap == 'DisableWorkspaceOutboundAccessProtection':
            kind_list = ITEM_KINDS_OAP_DISABLED
        else:
            kind_list = ITEM_KINDS_ANY
        kind = kind_list[rng.randrange(len(kind_list))]

    return {
        'type': ev_type,
        'datacontenttype': 'application/json',
        'id': str(uuid.uuid4()),
        'time': datetime.now(timezone.utc).isoformat(),
        'dataschemaversion': 1.0,
        'subject': f"/workspaces/{ws['WorkspaceId']}/items/{item_id}",
        'source': ws['WorkspaceId'],
        'specversion': 1.0,
        'data': {
            'workspaceId': ws['WorkspaceId'],
            'workspaceName': ws['WorkspaceName'],
            'itemName': f'stream_item_{item_id}',
            'itemId': item_id,
            'itemKind': kind,
            'executingPrincipalId': str(uuid.uuid4()),
            'executingPrincipalType': 'User',
        },
    }

session = requests.Session()
session.headers.update(ingest_headers)

def send_batch(count):
    if count <= 0:
        return 0
    flagged_extra = 1 if rng.random() < FlaggedProb else 0
    total = count + flagged_extra

    # NDJSON: one compact JSON object per line
    lines = [json.dumps(new_event(), separators=(',', ':')) for _ in range(count)]
    for _ in range(flagged_extra):
        lines.append(json.dumps(new_event(force_kind='Notebook'), separators=(',', ':')))
    payload = '\n'.join(lines) + '\n'

    try:
        r = session.post(ingest_uri, data=payload.encode('utf-8'), timeout=60)
        r.raise_for_status()
        return total
    except Exception as e:
        print(f'Ingest batch failed: {e}')
        return 0

## Main loop (runs for `DurationMinutes`)

In [ ]:
total_seconds = DurationMinutes * 60
start_utc     = datetime.now(timezone.utc)
end_utc       = start_utc + timedelta(seconds=total_seconds)
total_sent    = 0
sec_count     = 0

print(f'Profile: {LoadProfile}  Duration: {DurationMinutes}m  BatchSize: {BatchSize}')

while datetime.now(timezone.utc) < end_utc:
    sec_start = time.monotonic()

    # Resolve target EPS for this second
    if LoadProfile == 'steady':
        eps = RatePerSecond
    elif LoadProfile == 'burst':
        cyc = BurstSeconds + IdleSeconds
        eps = BurstRate if (sec_count % cyc) < BurstSeconds else 0
    elif LoadProfile == 'ramp':
        progress = (datetime.now(timezone.utc) - start_utc).total_seconds() / total_seconds
        eps = int(RatePerSecond + (BurstRate - RatePerSecond) * progress)
    else:
        raise ValueError(f'Unknown LoadProfile: {LoadProfile}')

    # Send in batches inside this 1-second window
    remaining = eps
    sent_this_sec = 0
    while remaining > 0:
        this_batch = min(BatchSize, remaining)
        sent_this_sec += send_batch(this_batch)
        remaining -= this_batch
    total_sent += sent_this_sec
    sec_count  += 1

    if sec_count % 10 == 0:
        elapsed = (datetime.now(timezone.utc) - start_utc).total_seconds()
        avg_eps = int(total_sent / max(elapsed, 1))
        print(f'  t={int(elapsed):5}s  target={eps:5} EPS  sent_last_sec={sent_this_sec:5}  total={total_sent:9}  avg={avg_eps:5} EPS')

    # Pace to ~1 second per iteration
    sleep = 1.0 - (time.monotonic() - sec_start)
    if sleep > 0:
        time.sleep(sleep)

elapsed_total = (datetime.now(timezone.utc) - start_utc).total_seconds()
avg = int(total_sent / max(elapsed_total, 1))
print(f'\nStream complete. Sent {total_sent} events in {elapsed_total:.0f}s ({avg} avg EPS).')